# 예제 IV · Post-Training Quantization

사전 학습된 **CIFAR-10 ResNet-20**의 Weight와 실제 ReLU Activation에 양자화·복원을 적용합니다. 상단에서 **Weight와 Activation을 독립적으로 설정**하고, 선택한 조합별로 복원 오차와 테스트 정확도를 비교합니다.

**기본 설정**은 Weight 8-bit / Symmetric / Uniform / Per-channel, Activation 8-bit / Asymmetric / Uniform / Per-tensor입니다. 노트북은 양자화 후 복원한 값을 부동소수점으로 계산하는 **Q/DQ 시뮬레이션**입니다.

**코드 셀 1 · Weight와 Activation 양자화 설정**

Weight와 Activation의 비트 수, 범위 방식, Grid와 Granularity를 지정하고 Calibration 128장과 테스트 512장을 설정합니다.

먼저 셀 7의 미리보기에서 설정을 하나씩 비교하고, 셀 9의 정확도와 누적표에서 모델 결과를 확인합니다.

In [ ]:
# @title 실습 설정
# Weight의 범위 방식, 눈금 간격, 변환 단위를 정한다. Group Size는 Per-group에 적용된다.
WEIGHT_BITS = 8 # @param {type:"integer"}
WEIGHT_SCHEME = "symmetric" # @param ["symmetric", "asymmetric"]
WEIGHT_GRID = "uniform" # @param ["uniform", "non-uniform"]
WEIGHT_GRANULARITY = "per-channel" # @param ["per-tensor", "per-channel", "per-group"]
WEIGHT_GROUP_SIZE = 16 # @param {type:"integer"}

# Activation은 Weight와 별도로 설정한다. Calibration에 선택한 변환 단위가 적용된다.
ACTIVATION_BITS = 8 # @param {type:"integer"}
ACTIVATION_SCHEME = "asymmetric" # @param ["symmetric", "asymmetric"]
ACTIVATION_GRID = "uniform" # @param ["uniform", "non-uniform"]
ACTIVATION_GRANULARITY = "per-tensor" # @param ["per-tensor", "per-channel", "per-group"]
ACTIVATION_GROUP_SIZE = 32 # @param {type:"integer"}

# μ는 Non-uniform에서 로그 압축의 강도를 정한다. 아래 영상 수는 수업 중 실행 시간을 조절한다.
MU = 15.0 # @param {type:"number"}
CALIBRATION_SAMPLES = 128 # @param {type:"integer"}
TEST_SAMPLES = 512 # @param {type:"integer"}

## 1. 옵션의 의미

| 설정 | 선택지 | 어떤 값이 바뀌는가 |
|---|---|---|
| 범위 | Symmetric / Asymmetric | Symmetric은 실수 0을 중심으로 좌우 범위를 같게 둡니다. Asymmetric은 관측한 음수·양수 범위를 따로 반영합니다. |
| 복원값 간격 | Uniform / Non-uniform | Uniform은 일정 간격을 사용합니다. Non-uniform은 이 실습의 μ-law companding으로 0 부근에 복원값을 촘촘히 배치합니다. |
| Granularity | Per-tensor / Per-channel / Per-group | 텐서 전체, 채널별, 채널 안의 그룹별로 변환 기준을 따로 정합니다. |
| 비트 수 및 그룹 크기 | Weight·Activation 개별 설정 | 비트 수가 정수 코드의 범위를 정합니다. 그룹 크기는 Per-group에서만 사용합니다. |

**Weight**의 Per-channel은 **출력 채널별** 기준입니다. Per-group에서는 각 출력 채널의 Weight를 펼쳐 연속 원소를 묶습니다. **Activation**의 Per-group에서는 각 Feature Map 채널의 공간 위치(H×W)를 펼쳐 묶고, Calibration 영상의 같은 위치 묶음에서 범위를 관측합니다.

Non-uniform의 Asymmetric 설정은 관측 범위에 따라 0 아래와 위에 사용할 코드 수를 나눕니다. [μ-law 함수 참고](https://docs.pytorch.org/audio/stable/generated/torchaudio.transforms.MuLawEncoding.html)

**코드 셀 2 · 실행 장치와 캐시 설정**

PyTorch와 torchvision을 불러오고 실습에 사용할 GPU 또는 CPU, 시드, 데이터와 가중치 저장 경로를 설정합니다.

In [ ]:
# 실습 설정을 한곳에서 관리한다. 데이터·가중치 캐시를 재사용하고 CUDA가 있으면 GPU로 실행한다.
import copy
import os
import time
from itertools import islice
from pathlib import Path
from statistics import mean

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 동일한 시드와 배치 수를 사용해 반복 실행 조건을 맞춘다.
SEED = 50
BATCH_SIZE = 64
DATA_ROOT = Path(os.environ.get("AX_LAB_DATA_ROOT", "./data"))
ASSET_ROOT = Path(os.environ.get("AX_LAB_CACHE", "./lab_cache"))
ASSET_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("실행 장치:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU")

## 2. 사전 학습 모델과 데이터 준비

사전 학습 Weight를 불러와 ResNet-20을 평가 모드로 둡니다. 코드의 두 Loader는 역할이 다릅니다.

- **Calibration 128장** · 학습 분할에서 뽑은 대표 영상. Activation 범위를 정하는 데 사용
- **테스트 512장** · 테스트 분할에서 뽑은 고정 영상. FP32와 선택한 Q/DQ 모델의 정확도 비교에 사용

데이터와 체크포인트가 캐시에 있으면 다음 실행에서 재사용합니다.

**코드 셀 3 · 모델 다운로드 함수 정의**

고정된 GitHub 커밋의 소스와 가중치를 내려받아 캐시에 저장하고 ZIP의 항목 경로를 확인한 뒤 압축을 푸는 함수를 정의합니다.

In [ ]:
from urllib.request import urlretrieve
from zipfile import ZipFile
import sys

# 체크포인트와 소스를 캐시에 보관한다. 다운로드가 완료되면 임시 파일을 정상 파일 이름으로 바꾼다.
def fetch(url, filename):
    destination = ASSET_ROOT / filename
    if not destination.is_file():
        pending = destination.with_suffix(destination.suffix + ".download")
        try:
            urlretrieve(url, pending)
            pending.replace(destination)
        finally:
            pending.unlink(missing_ok=True)
    return destination

# 고정한 커밋의 ZIP을 받고, 각 항목의 경로를 검사한 뒤 추출한다.
def fetch_repository(owner, repository, revision, filename):
    archive = fetch(f"https://codeload.github.com/{owner}/{repository}/zip/{revision}", filename)
    root = ASSET_ROOT / f"{repository}-{revision}"
    if not root.exists():
        with ZipFile(archive) as source_zip:
            for entry in source_zip.infolist():
                destination = (ASSET_ROOT / entry.filename).resolve()
                if not destination.is_relative_to(root.resolve()):
                    raise ValueError(f"Unexpected archive path: {entry.filename}")
            source_zip.extractall(ASSET_ROOT)
    return root.resolve()

**코드 셀 4 · ResNet-20과 Calibration 데이터 준비**

CIFAR-10으로 사전 학습한 ResNet-20을 불러오고 학습 분할의 Calibration 영상과 별도 테스트 분할의 평가 영상을 준비합니다.

이미 내려받은 CIFAR-10은 런타임의 캐시에서 읽습니다. 첫 실행에서는 공유 Google Drive에서 받고, 실패하면 CIFAR-10의 공식 주소를 사용합니다.

In [ ]:
import matplotlib.pyplot as plt

# 고정한 소스의 ResNet-20 구조에 공개 CIFAR-10 학습 가중치를 로드한다.
RESNET20_REVISION = "786c16252c0fc58ee9adac063f8337cc4a7a497a"
resnet_source = fetch_repository("chenyaofo", "pytorch-cifar-models", RESNET20_REVISION, "resnet-cifar.zip")
sys.path.insert(0, str(resnet_source))
from pytorch_cifar_models import cifar10_resnet20

model = cifar10_resnet20(pretrained=False)
weights_url = ("https://github.com/chenyaofo/pytorch-cifar-models/releases/download/"
               "resnet/cifar10_resnet20-4118986f.pt")
weights_path = fetch(weights_url, "resnet20.pt")
model.load_state_dict(torch.load(weights_path, map_location="cpu", weights_only=True))
model.to(DEVICE).eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
# 학습 분할의 대표 영상으로 Calibration 범위를 정하고, 별도 테스트 분할에서 정확도를 비교한다.
from http.client import HTTPException
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlopen

from torchvision.datasets.utils import check_integrity


class DriveCIFAR10(datasets.CIFAR10):
    # 압축 해제된 데이터나 검증된 압축 파일을 먼저 재사용한다.
    def download(self):
        if self._check_integrity():
            return

        root = Path(self.root)
        archive = root / self.filename
        if not check_integrity(archive, self.tgz_md5):
            drive_url = (
                "https://drive.usercontent.google.com/download"
                "?id=1Sx382Bt_XcAUj9_igoCgWoNQ6c5ooSa6&export=download&confirm=t"
            )
            try:
                root.mkdir(parents=True, exist_ok=True)
                with TemporaryDirectory(dir=root) as temp_dir:
                    staged = Path(temp_dir) / self.filename
                    print("Google Drive에서 CIFAR-10을 내려받습니다.")
                    with urlopen(drive_url, timeout=30) as response:
                        if response.headers.get_content_type() == "text/html":
                            raise ValueError("Drive가 파일 대신 안내 페이지를 반환했습니다.")
                        with staged.open("wb") as output:
                            copyfileobj(response, output, length=1024 * 1024)
                    if not check_integrity(staged, self.tgz_md5):
                        raise ValueError("CIFAR-10 원본 체크섬과 일치하지 않습니다.")
                    staged.replace(archive)
            except (OSError, HTTPException, ValueError):
                print("Drive에서 CIFAR-10을 받지 못해 공식 다운로드 주소로 다시 시도합니다.")

        # torchvision이 원본 체크섬을 확인하고 필요한 경우 압축을 해제한다.
        super().download()


calibration_data = DriveCIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_data = DriveCIFAR10(DATA_ROOT, train=False, download=True, transform=transform)

if not 1 <= CALIBRATION_SAMPLES <= len(calibration_data):
    raise ValueError("CALIBRATION_SAMPLES를 학습 분할 안에서 1장 이상으로 선택하세요.")
if not 1 <= TEST_SAMPLES <= len(test_data):
    raise ValueError("TEST_SAMPLES를 테스트 분할 안에서 1장 이상으로 선택하세요.")

indices = torch.randperm(len(calibration_data), generator=torch.Generator().manual_seed(SEED))
eval_indices = torch.randperm(len(test_data), generator=torch.Generator().manual_seed(SEED + 1))
calibration_loader = DataLoader(Subset(calibration_data, indices[:CALIBRATION_SAMPLES].tolist()), batch_size=BATCH_SIZE)
test_loader = DataLoader(Subset(test_data, eval_indices[:TEST_SAMPLES].tolist()), batch_size=BATCH_SIZE)
print(f"Calibration {len(calibration_loader.dataset)}장 / 테스트 {len(test_loader.dataset)}장")

## 3. 그룹별 범위와 양자화·복원 함수

1. `tensor_blocks()`가 변환 기준을 공유할 값들을 한 행으로 모읍니다. 변환 후에는 다시 원래 Tensor 모양으로 돌려놓습니다.
2. `quantization_parameters()`가 비트 수와 관측 범위로부터 정수 코드 범위와 Zero-point를 정합니다. Uniform에서는 Scale이 실수의 눈금 간격입니다.
3. `quantize_dequantize()`는 선택한 눈금에 값을 대응시키고 실수값으로 복원합니다. Non-uniform에서는 로그 압축과 그 역함수를 사용합니다.

Weight의 범위는 Weight 자체에서 계산합니다. Activation의 범위는 다음 단계에서 관측해 고정합니다.

**코드 셀 5 · 양자화와 복원 함수 정의**

Weight와 Activation의 설정을 검증하고, 선택한 범위 방식과 Grid 및 Granularity에 따라 값을 양자화한 뒤 실수로 복원하는 Q/DQ 함수를 정의합니다.

In [ ]:
from dataclasses import dataclass, replace
import math

# 설정값을 하나로 묶고 비트 수와 선택지가 지원 범위를 벗어나지 않았는지 확인한다.
@dataclass(frozen=True)
class QuantConfig:
    bits: int
    scheme: str
    grid: str
    granularity: str
    group_size: int
    mu: float = 15.0

    def __post_init__(self):
        if not 2 <= self.bits <= 8:
            raise ValueError("비트 수는 2~8 사이로 설정하세요.")
        if self.scheme not in {"symmetric", "asymmetric"}:
            raise ValueError(f"지원하지 않는 scheme: {self.scheme}")
        if self.grid not in {"uniform", "non-uniform"}:
            raise ValueError(f"지원하지 않는 grid: {self.grid}")
        if self.granularity not in {"per-tensor", "per-channel", "per-group"}:
            raise ValueError(f"지원하지 않는 granularity: {self.granularity}")
        if self.group_size < 1 or not math.isfinite(self.mu) or self.mu <= 0:
            raise ValueError("Group size와 μ는 양수여야 합니다.")

weight_cfg = QuantConfig(WEIGHT_BITS, WEIGHT_SCHEME, WEIGHT_GRID,
                         WEIGHT_GRANULARITY, WEIGHT_GROUP_SIZE, MU)
activation_cfg = QuantConfig(ACTIVATION_BITS, ACTIVATION_SCHEME, ACTIVATION_GRID,
                             ACTIVATION_GRANULARITY, ACTIVATION_GROUP_SIZE, MU)

# Scale과 Zero-point를 함께 사용할 값들을 행으로 만든다. 변환 후 원래 축으로 되돌리는 함수도 반환한다.
def tensor_blocks(x, config, activation=False):
    # 블록 행마다 하나의 독립적인 변환 기준을 적용합니다.
    if config.granularity == "per-tensor":
        return x.reshape(1, -1), lambda blocks: blocks.reshape_as(x)

    rows = x.movedim(1, 0) if activation else x
    channels = rows.shape[0]
    # Per-channel에서는 한 채널마다 별도 변환 기준을 쓴다. Activation의 채널 축은 앞에 옮겨 둔다.
    if config.granularity == "per-channel":
        def restore(blocks):
            output = blocks.reshape_as(rows)
            return output.movedim(0, 1) if activation else output
        return rows.reshape(channels, -1), restore

    if activation:
        samples = x.shape[0]
        rows_by_sample = rows.reshape(channels, samples, -1)
    else:
        samples = 1
        rows_by_sample = rows.reshape(channels, 1, -1)

    length = rows_by_sample.shape[-1]
    group_size = config.group_size
    groups = (length + group_size - 1) // group_size
    # 범위는 항상 0을 포함하도록 정하므로 끝 그룹의 0 padding은 기준을 바꾸지 않습니다.
    # Per-group에서도 마지막 원소를 포함하도록 0을 채우고, 같은 공간 그룹을 영상끼리 묶는다.
    padded = F.pad(rows_by_sample, (0, groups * group_size - length))
    blocks = padded.reshape(channels, samples, groups, group_size)
    blocks = blocks.permute(0, 2, 1, 3).reshape(channels * groups, samples * group_size)

    def restore(quantized):
        result = quantized.reshape(channels, groups, samples, group_size)
        result = result.permute(0, 2, 1, 3).reshape(channels, samples, -1)
        result = result[..., :length].reshape_as(rows)
        return result.movedim(0, 1) if activation else result

    return blocks, restore

# 관측 범위에 0을 포함시킨다. Symmetric은 절댓값 최댓값 기준, Asymmetric은 전체 범위 폭과 Zero-
# point 기준이다.
def quantization_parameters(low, high, config):
    low = low.clamp_max(0)
    high = high.clamp_min(0)
    if config.scheme == "symmetric":
        qmax = 2 ** (config.bits - 1) - 1
        peak = torch.maximum(-low, high)
        scale = torch.where(peak > 0, peak / qmax, 1.0)
        zero = torch.zeros_like(scale)
        return -peak, peak, scale, zero, -qmax, qmax

    qmax = 2**config.bits - 1
    width = high - low
    scale = torch.where(width > 0, width / qmax, 1.0)
    zero = (-low / scale).round().clamp(0, qmax)
    return low, high, scale, zero, 0, qmax

def expand_companded(code_steps, levels, span, mu):
    # 정수 코드를 μ-law의 역함수로 복원한다.
    fraction = code_steps / levels.clamp_min(1)
    return torch.expm1(fraction * math.log1p(mu)) * (span / mu)


# Weight는 현재 범위에서, Activation은 Calibration에서 고정한 범위로 양자화·복원한다.
def quantize_dequantize(x, config, activation=False, bounds=None):
    blocks, restore = tensor_blocks(x, config, activation)
    low, high = blocks.aminmax(dim=1, keepdim=True) if bounds is None else bounds
    lo, hi, scale, zero, qmin, qmax = quantization_parameters(low, high, config)
    if config.grid == "uniform":
        codes = (blocks / scale).round().add(zero).clamp(qmin, qmax)
        return restore((codes - zero) * scale)

    # μ-law companding: 음수·양수 영역을 각자의 끝값으로 정규화한 뒤 복원합니다.
    # Non-uniform은 양·음 영역을 0 기준으로 나눠 μ-law로 압축한다. 복원하면 0 근처 눈금이 촘촘해진다.
    negative = blocks < 0
    span = torch.where(negative, -lo, hi)
    levels = torch.where(negative, zero - qmin, qmax - zero)
    safe_span = torch.where(span > 0, span, 1.0)
    normalized = blocks.abs().div(safe_span).clamp(0, 1)
    normalized = torch.where(span > 0, normalized, 0.0)
    log_mu = math.log1p(config.mu)
    compressed = torch.log1p(config.mu * normalized) / log_mu
    # 압축 영역에서 정수 단계로 반올림한 뒤 역함수로 실수 크기를 복원한다.
    code_steps = (compressed * levels).round()
    reconstructed = expand_companded(code_steps, levels, span, config.mu)
    return restore(torch.where(negative, -reconstructed, reconstructed))


def format_quant_config(config):
    parts = [f"{config.bits}-bit", config.scheme, config.grid, config.granularity]
    if config.granularity == "per-group":
        parts.append(f"group={config.group_size}")
    if config.grid == "non-uniform":
        parts.append(f"μ={config.mu:g}")
    return " · ".join(parts)

## 4. Activation Calibration

대표 영상을 FP32 ResNet-20에 입력하고 ReLU 출력이 관측될 때마다 Min/Max를 모읍니다. 평가 때는 이 범위를 재사용합니다.

**첫 ReLU의 범위 개수**를 확인합니다. Per-tensor에서 한 개이며, Per-channel 또는 Per-group에서는 선택한 채널·그룹 수만큼 범위를 관리합니다. 이 숫자는 어떤 값들이 같은 변환 기준을 쓰는지 확인하는 단서입니다.

**코드 셀 6 · ReLU Activation 범위 측정**

Calibration 영상을 통과시키며 각 ReLU의 최솟값과 최댓값을 수집하고 첫 ReLU의 Activation 예시를 저장합니다.

In [ ]:
# ReLU Hook으로 각 배치의 범위를 수집하고, 전체 Calibration 동안의 Min/Max를 합친다.
@torch.inference_mode()
def calibrate_model(model, loader, config, capture_sample=False, relu_name=None):
    ranges = {}
    sample = {}

    def collect(name):
        def update(_module, _inputs, output):
            blocks, _ = tensor_blocks(output, config, activation=True)
            lo, hi = blocks.aminmax(dim=1, keepdim=True)
            if name in ranges:
                previous_lo, previous_hi = ranges[name]
                lo = torch.minimum(previous_lo, lo)
                hi = torch.maximum(previous_hi, hi)
            ranges[name] = (lo, hi)
            if capture_sample and not sample:
                sample.update(name=name, activation=output[:1].detach().clone())
        return update

    # 실행 중 필요한 계층에만 Hook을 연결하고 블록이 끝나면 제거한다.
    hooks = [
        module.register_forward_hook(collect(name))
        for name, module in model.named_modules()
        if isinstance(module, nn.ReLU) and (relu_name is None or name == relu_name)
    ]
    try:
        model.eval()
        for images, _ in loader:
            model(images.to(DEVICE))
    finally:
        for hook in hooks:
            hook.remove()
    return ranges, sample

# 선택한 Activation 설정으로 범위를 수집하고, 분포를 그릴 실제 샘플을 보관한다.
ranges, sample = calibrate_model(model, calibration_loader, activation_cfg, capture_sample=True)
print(f"Activation 범위를 측정한 ReLU: {len(ranges)}개")
print(f"첫 ReLU의 범위 개수: {ranges[sample['name']][0].shape[0]}")

## 5. 선택한 설정 미리보기

**코드 셀 7 · 실제 Weight와 Activation의 설정 비교**

셀 7의 오른쪽 메뉴에서 **관찰 대상**(Weight 또는 Activation)과 **비교 옵션**을 고릅니다. 화면을 바꿀 때는 셀 7만 다시 실행합니다. 셀 1의 평가 설정을 바꾼 경우에는 **모두 실행**합니다.

- **Bits · Scheme · Grid · Mu**는 같은 첫 Conv2d Weight 또는 첫 ReLU Activation에서 원본값과 복원값, 눈금 간격과 개수를 보여줍니다. Bits는 첫 ReLU의 예시 원소를 점선과 점으로 표시합니다. Grid는 같은 범위에서 Uniform과 Non-uniform의 μ=5, μ=15를 비교합니다. Mu에서는 셀 1에서 선택한 μ를 다른 값과 비교합니다.
- **Granularity · Group Size**는 처음 네 채널에서 범위를 공유하는 위치를 색으로 표시합니다. 그림에 보이는 범위 기준 개수와 전체 텐서의 범위 기준 개수를 각각 표시합니다. 색은 여러 그룹에서 반복될 수 있습니다. Activation에서 범위를 공유하는 단위가 바뀌면 해당 조건의 Calibration을 다시 계산합니다.
- **Calibration Samples**는 같은 첫 ReLU 단위의 관측 범위와, 고정된 테스트 이미지 한 장에서 그 범위 밖에 놓인 값의 비율을 비교합니다.
- **Test Samples**를 선택하면 셀 9에서 같은 Q/DQ 모델을 다른 장수로 평가합니다. 차트에는 각 테스트 장수와 정확도가 표시됩니다.

**강의자료의 Activation 미리보기 조건**

관찰 대상을 **Activation**으로 두고 Calibration 128장, 4-bit, asymmetric, uniform, per-tensor를 사용합니다. 이 조건에서 **Bits**는 같은 첫 ReLU 원소의 8-bit와 4-bit 복원값을 보여줍니다. **Scheme**은 4-bit에서 0 이상 눈금 수를, **Grid**는 같은 4-bit 관측 범위에서 세 눈금 간격을 비교합니다.

**Granularity**에서는 Activation 그룹 크기를 32로 둡니다. 처음 네 채널에 보이는 범위 기준은 per-tensor 1개, per-channel 4개, per-group 128개이며, 전체는 각각 1개, 16개, 512개입니다. **Group Size**를 볼 때는 셀 1에서 Activation을 per-group으로 바꿔 모두 실행한 뒤, 셀 7에서 그룹 크기 32와 16을 비교합니다.

In [ ]:
# @title 현재 설정 미리보기
PREVIEW_TARGET = "Activation" # @param ["Weight", "Activation"]
PREVIEW_OPTION = "Bits" # @param ["Bits", "Scheme", "Grid", "Mu", "Granularity", "Group Size", "Calibration Samples", "Test Samples"]

# 셀 5·6에서 실제로 적용한 설정을 표시한다. 미리보기의 비교 설정은 평가에 저장하지 않는다.
print("Weight:", format_quant_config(weight_cfg))
print("Activation:", format_quant_config(activation_cfg))
print(f"Calibration {len(calibration_loader.dataset):,}장 / 테스트 {len(test_loader.dataset):,}장")


def layout_key(config):
    # Per-group에서만 그룹 크기가 범위의 형태를 바꾼다.
    size = config.group_size if config.granularity == "per-group" else None
    return config.granularity, size


@torch.inference_mode()
def preview_bounds(config, loader):
    if layout_key(config) == layout_key(activation_cfg) and loader is calibration_loader:
        return ranges[sample["name"]]
    observed, _ = calibrate_model(model, loader, config, relu_name=sample["name"])
    return observed[sample["name"]]


def preview_options(config, option):
    # 같은 데이터와 나머지 설정을 고정한 채 아래 항목 하나만 바꾼다.
    if option == "Bits":
        bits = (8, 4) if config.bits == 8 else (8, config.bits)
        return [replace(config, bits=value) for value in bits], [f"{value}-bit" for value in bits]
    if option == "Scheme":
        values = ("symmetric", "asymmetric")
        return [replace(config, scheme=value) for value in values], list(values)
    if option == "Grid":
        return [
            replace(config, grid="uniform"),
            replace(config, grid="non-uniform", mu=5.0),
            replace(config, grid="non-uniform", mu=15.0),
        ], ["Uniform", "Non-uniform (μ=5)", "Non-uniform (μ=15)"]
    if option == "Mu":
        values = (config.mu, 15 if config.mu == 5 else 5)
        return [replace(config, mu=value) for value in values], [f"μ={value:g}" for value in values]
    if option == "Granularity":
        values = ("per-tensor", "per-channel", "per-group")
        return [replace(config, granularity=value) for value in values], list(values)
    sizes = (config.group_size, 32 if config.group_size == 16 else 16)
    return [replace(config, group_size=size) for size in sizes], [f"group={size}" for size in sizes]


def reconstruction_levels(config, low, high):
    # 실제 관측 범위에 맞는 모든 정수 코드를 복원해 눈금의 위치를 구한다.
    lo, hi, scale, zero, qmin, qmax = quantization_parameters(low, high, config)
    offsets = torch.arange(qmin, qmax + 1, device=lo.device, dtype=lo.dtype) - zero
    if config.grid == "uniform":
        return (offsets * scale).unique(sorted=True)
    negative = offsets < 0
    span = torch.where(negative, -lo, hi)
    levels = torch.where(negative, zero - qmin, qmax - zero)
    decoded = expand_companded(offsets.abs(), levels, span, config.mu)
    return torch.where(negative, -decoded, decoded).unique(sorted=True)


@torch.inference_mode()
def show_mapping(raw, is_activation, configs, labels, title):
    blocks, restore_layout = tensor_blocks(raw, configs[0], is_activation)
    if is_activation:
        low, high = preview_bounds(configs[0], calibration_loader)
    else:
        low, high = blocks.aminmax(dim=1, keepdim=True)

    # Bits 예시의 좌표는 실제 첫 ReLU 텐서의 배치·채널·행·열 순서다.
    example_index = (0, 13, 4, 3) if is_activation and PREVIEW_OPTION == "Bits" else None
    if example_index is None:
        unit = int(blocks.abs().amax(dim=1).argmax().item())
    else:
        unit_ids = torch.arange(blocks.size(0), device=raw.device).view(-1, 1).expand_as(blocks)
        unit = int(restore_layout(unit_ids)[example_index].item())
        example_input = raw[example_index].item()
        print(f"예시 원소 NCHW {example_index} | 실제 입력값 {example_input:.4f}")
    x = blocks[unit]
    positions = torch.linspace(0, x.numel() - 1, min(1500, x.numel()), device=x.device).round().long()
    visible_values = x[positions].cpu().numpy()
    colors = ("#475569", "#167d9a", "#b45309")

    fig = plt.figure(figsize=(9, 5.8 if PREVIEW_OPTION in ("Grid", "Mu") else 5),
                     constrained_layout=True)
    zoom_grid = PREVIEW_OPTION in ("Grid", "Mu")
    layout = fig.add_gridspec(3 if zoom_grid else 2, 1,
                              height_ratios=[3, 1, 1] if zoom_grid else [3, 1])
    mapping = fig.add_subplot(layout[0])
    ruler = fig.add_subplot(layout[1], sharex=mapping)
    zoom = fig.add_subplot(layout[2]) if zoom_grid else None
    lo, hi = min(x.min().item(), low[unit].item()), max(x.max().item(), high[unit].item())
    mapping.plot([lo, hi], [lo, hi], "--", color="#999999", linewidth=1, label="FP32 (y = x)")
    row_labels = list(labels)
    if example_index is not None:
        for axis in (mapping, ruler):
            axis.axvline(example_input, color="#222222", linestyle=":", linewidth=1.1)

    for index, (config, label, color) in enumerate(zip(configs, labels, colors)):
        bounds = (low, high)
        restored = quantize_dequantize(raw, config, activation=is_activation, bounds=bounds)
        restored_blocks, _ = tensor_blocks(restored, config, is_activation)
        result = restored_blocks[unit]
        levels = reconstruction_levels(config, low[unit, 0], high[unit, 0])
        mapping.scatter(visible_values, result[positions].cpu().numpy(),
                        s=8, alpha=0.58, color=color, label=label)
        positions_on_ruler = levels.cpu().numpy()
        ruler.vlines(positions_on_ruler, index - 0.28, index + 0.28, color=color, linewidth=0.9)
        if zoom is not None:
            zoom.vlines(positions_on_ruler, index - 0.28, index + 0.28, color=color, linewidth=1.0)
        summary = [label, f"범위 기준 {unit + 1}번", f"복원 눈금 {levels.numel()}개"]
        if example_index is not None:
            value = restored[example_index].item()
            mapping.scatter(example_input, value, s=36, color=color, edgecolor="white", zorder=5)
            ruler.scatter(value, index, s=32, color=color, edgecolor="white", zorder=5)
            row_labels[index] = f"{label} → {value:.4f}"
            summary.append(f"예시 복원값 {value:.4f}")
        if is_activation and PREVIEW_OPTION == "Scheme":
            nonnegative_count = int((levels >= 0).sum().item())
            row_labels[index] = f"{label} ({nonnegative_count} levels ≥ 0)"
            summary.append(f"0 이상 눈금 {nonnegative_count}개")
        if PREVIEW_OPTION in ("Grid", "Mu"):
            positive_levels = levels[levels > 0]
            if positive_levels.numel():
                summary.append(f"0에서 첫 양의 눈금까지 {positive_levels[0].item():.3f}")
        summary.append(f"이 단위 MAE {(x - result).abs().mean().item():.6f}")
        print(" | ".join(summary))

    print(f"같은 단위의 관측 범위 [{low[unit].item():.6g}, {high[unit].item():.6g}]")
    mapping.set(title=title, ylabel="Reconstructed value")
    mapping.legend()
    ruler.axvline(0, color="#999999", linestyle="--", linewidth=0.8)
    ruler.set(yticks=range(len(row_labels)), yticklabels=row_labels, xlabel="Value on the FP32 scale")
    ruler.set_title("Available reconstruction levels", loc="left")
    ruler.set_ylim(len(row_labels) - 0.45, -0.55)
    # 눈금과 관측 범위를 함께 표시한다. 비대칭/대칭의 차이가 x축 밖으로 잘리지 않게 한다.
    ruler.margins(x=0.045)
    if zoom is not None:
        extent = max(abs(lo), abs(hi), 1e-6) * 0.12
        zoom.set(xlim=(-extent, extent) if lo < 0 else (0, extent),
                 yticks=range(len(labels)), yticklabels=labels, xlabel="Value near zero (zoom)")
        zoom.set_ylim(len(labels) - 0.45, -0.55)
    plt.show()


@torch.inference_mode()
def show_groups(raw, is_activation, configs, labels, title):
    fig, axes = plt.subplots(1, len(configs), figsize=(10, 3.6), constrained_layout=True)
    for axis, config, label in zip(axes, configs, labels):
        blocks, restore = tensor_blocks(raw, config, is_activation)
        groups = torch.arange(blocks.size(0), device=raw.device).reshape(-1, 1).expand_as(blocks)
        layout = restore(groups)
        channels = layout[0] if is_activation else layout.flatten(1, 2)
        # 실제 텐서의 처음 네 채널을 같은 순서로 이어서 그린다.
        shown = channels[:4]
        width = shown.size(-1)
        mask = torch.cat(tuple(shown), dim=1).cpu().numpy()
        axis.imshow(mask % 20, cmap="tab20", vmin=0, vmax=19,
                    interpolation="nearest", aspect="auto")
        for boundary in range(1, len(shown)):
            axis.axvline(boundary * width - 0.5, color="white", linewidth=1.2)
        total_count = blocks.size(0)
        visible_count = shown.unique().numel()
        axis.set(title=f"{label}\nShown: {visible_count:,}   Total: {total_count:,}", xticks=[], yticks=[])
        bounds = preview_bounds(config, calibration_loader) if is_activation else None
        result = quantize_dequantize(raw, config, activation=is_activation, bounds=bounds)
        summary = [label, f"그림 범위 기준 {visible_count:,}개", f"전체 범위 기준 {total_count:,}개"]
        if is_activation and PREVIEW_OPTION == "Group Size":
            first_row_count = channels[0, 0].unique().numel()
            summary.append(f"첫 채널 첫 행의 값 {width}개는 범위 기준 {first_row_count}개를 사용")
        summary.append(f"같은 텐서 MAE {(raw - result).abs().mean().item():.6f}")
        print(" | ".join(summary))
    fig.suptitle(f"{title} | first four channels")
    plt.show()


@torch.inference_mode()
def show_calibration_preview():
    # 평가 영상 한 장은 두 Calibration 조건에서 고정한다.
    image, _ = test_loader.dataset[0]
    probe = {}
    first_relu = dict(model.named_modules())[sample["name"]]
    handle = first_relu.register_forward_hook(
        lambda _module, _inputs, output: probe.update(activation=output.detach().clone())
    )
    try:
        model(image.unsqueeze(0).to(DEVICE))
    finally:
        handle.remove()

    raw = probe["activation"]
    blocks, _ = tensor_blocks(raw, activation_cfg, activation=True)
    unit = int(blocks.abs().amax(dim=1).argmax().item())
    current_count = len(calibration_loader.dataset)
    other_count = 128 if current_count != 128 else 32
    other_loader = DataLoader(Subset(calibration_data, indices[:other_count].tolist()), batch_size=BATCH_SIZE)

    fig, axis = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
    for position, loader in enumerate((calibration_loader, other_loader)):
        lows, highs = preview_bounds(activation_cfg, loader)
        low, high = lows[unit].item(), highs[unit].item()
        x = blocks[unit]
        outside = ((x < low) | (x > high)).float().mean().item()
        axis.hlines(position, low, high, color=("#475569", "#167d9a")[position], linewidth=6)
        axis.scatter([low, high], [position, position], color="#141414", s=20, zorder=3)
        axis.annotate(f"{low:.4g}", (low, position), xytext=(3, 7), textcoords="offset points", ha="left")
        axis.annotate(f"{high:.4g}", (high, position), xytext=(-3, 7), textcoords="offset points", ha="right")
        print(f"Calibration {len(loader.dataset):,}장 | 관측 범위 [{low:.6g}, {high:.6g}] | "
              f"같은 테스트 영상에서 범위 밖 값 {outside:.2%}")
    axis.set(title=f"First ReLU | observed range for unit {unit + 1}",
             xlabel="Activation value",
             yticks=[0, 1], yticklabels=[f"{current_count:,} images", f"{other_count:,} images"])
    axis.set_ylim(-0.5, 1.5)
    plt.show()


def preview_quantization():
    if PREVIEW_OPTION == "Calibration Samples":
        show_calibration_preview()
        return
    if PREVIEW_OPTION == "Test Samples":
        print("테스트 이미지 수만 비교합니다. 셀 9에서 같은 Q/DQ 모델의 두 정확도를 확인합니다.")
        return

    is_activation = PREVIEW_TARGET == "Activation"
    config = activation_cfg if is_activation else weight_cfg
    if PREVIEW_OPTION == "Mu" and config.grid != "non-uniform":
        print(f"μ를 비교하려면 셀 1의 {PREVIEW_TARGET.upper()}_GRID를 non-uniform으로 선택하고 모두 실행합니다.")
        return
    if PREVIEW_OPTION == "Group Size" and config.granularity != "per-group":
        print(f"그룹 크기를 비교하려면 셀 1의 {PREVIEW_TARGET.upper()}_GRANULARITY를 per-group으로 선택하고 모두 실행합니다.")
        return

    raw = sample["activation"] if is_activation else model.conv1.weight.detach()
    configs, labels = preview_options(config, PREVIEW_OPTION)
    title = "First ReLU Activation" if is_activation else "First Conv2d Weight"
    if PREVIEW_OPTION in ("Granularity", "Group Size"):
        show_groups(raw, is_activation, configs, labels, title)
    else:
        show_mapping(raw, is_activation, configs, labels, title)


preview_quantization()

## 6. 같은 CIFAR-10 테스트 샘플에서 평가

선택한 Weight·Activation 변환을 복제 모델에 적용한 뒤, 같은 영상에서 **FP32 정확도**, **Q/DQ 정확도**, **FP32 대비 차이(%p)**를 확인합니다.

설정의 영향을 비교할 때는 비트 수·범위·간격·Granularity 중 **한 항목만 바꾸고** 다른 항목과 테스트 샘플은 유지합니다. 설정 셀을 수정한 뒤 모두 실행하면 새 Activation 설정에 맞춰 Calibration도 다시 수행됩니다.

**코드 셀 8 · Q/DQ 적용과 평가 함수 정의**

복제한 모델의 Conv2d와 Linear Weight를 Q/DQ하고 평가 중에는 측정한 범위로 ReLU Activation을 Q/DQ하는 함수를 준비합니다.

In [ ]:
# 기준 모델을 복제하고 Conv2d와 Linear의 Weight에만 Q/DQ를 적용한다.
@torch.inference_mode()
def quantized_weight_model(model, config):
    modified = copy.deepcopy(model).eval()
    for module in modified.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            module.weight.copy_(quantize_dequantize(module.weight, config))
    return modified

# 평가 중에는 이미 정한 Activation 범위로 ReLU 출력에 Q/DQ를 적용한다.
def attach_activation_hooks(model, config, ranges):
    def make_hook(bounds):
        def apply(_module, _inputs, output):
            return quantize_dequantize(output, config, activation=True, bounds=bounds)
        return apply

    return [
        module.register_forward_hook(make_hook(ranges[name]))
        for name, module in model.named_modules() if name in ranges
    ]

# 평가 모드에서 각 영상의 최댓값 클래스를 정답과 비교해 전체 정확도를 집계한다.
@torch.inference_mode()
def evaluate(model, loader):
    correct, count = 0, 0
    model.eval()
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        count += labels.numel()
    return 100 * correct / count

**코드 셀 9 · FP32와 Q/DQ 정확도 비교**

FP32 모델과 Weight·Activation에 Q/DQ를 적용한 모델을 같은 테스트 영상으로 평가해 정확도와 변화 폭을 출력합니다.

상단 Colab 폼에서 값을 바꾼 뒤 **런타임 → 모두 실행**을 누릅니다. 셀 9까지 완료할 때마다 선택한 **Weight·Activation 설정**과 **첫 ReLU 복원 MAE, 테스트 정확도, FP32 대비 변화**가 같은 런타임의 표에 한 행씩 쌓입니다.

복원 MAE는 Calibration 이미지 한 장의 첫 ReLU 출력에서 측정합니다. 정확도 비교는 Calibration·테스트 영상 수를 고정하고 양자화 설정 한 항목만 바꾸어 진행합니다. Colab 런타임을 다시 시작하면 표는 새로 시작합니다.

**Test Samples 미리보기**에서는 같은 Q/DQ 설정에서 두 테스트 이미지 수의 정확도를 따로 표시합니다. 누적표에는 셀 1에서 선택한 현재 평가의 결과가 한 행 쌓입니다. 첫 ReLU MAE는 두 정확도와 별도로 Calibration에서 확보한 이미지 한 장으로 측정합니다.

**강의자료의 비교 기록 재현**

Weight를 8-bit, symmetric, uniform, per-channel로 두고 Calibration 128장과 테스트 512장을 고정합니다. Activation은 asymmetric와 uniform을 유지한 채 **8-bit per-tensor → 4-bit per-tensor → 4-bit per-channel** 순으로 설정하고 각각 모두 실행합니다. 마지막 두 행에서 비트 수는 같고 범위를 공유하는 단위만 달라집니다.

마지막 열의 이론적 Activation 비트량은 FP32 원소당 32-bit 대비 비율입니다. 8-bit는 25%, 4-bit는 12.5%입니다. 오차와 정확도는 Q/DQ로 복원한 값에서 계산합니다.

In [ ]:
start = time.perf_counter()
# 같은 테스트 샘플에서 FP32와 선택한 Q/DQ 모델을 평가한다.
simulated = quantized_weight_model(model, weight_cfg)
fp32_accuracy = evaluate(model, test_loader)
hooks = attach_activation_hooks(simulated, activation_cfg, ranges)
try:
    selected_accuracy = evaluate(simulated, test_loader)
finally:
    for hook in hooks:
        hook.remove()

# Calibration 이미지 한 장에서 첫 ReLU Activation의 복원 오차를 계산한다.
with torch.inference_mode():
    activation = sample["activation"]
    restored = quantize_dequantize(
        activation, activation_cfg, activation=True, bounds=ranges[sample["name"]]
    )
    selected_mae = (activation - restored).abs().mean().item()

print(f"FP32 정확도: {fp32_accuracy:.2f}%")
print(f"Q/DQ 정확도: {selected_accuracy:.2f}%")
print(f"FP32 대비 변화: {selected_accuracy - fp32_accuracy:+.2f}%p")
print(f"변환 및 비교 평가: {time.perf_counter() - start:.2f} s")


# 같은 Q/DQ 설정에서 평가 이미지 수만 바꾼다. 비교값은 누적 기록의 별도 실행으로 넣지 않는다.
if globals().get("PREVIEW_OPTION") == "Test Samples":
    current_count = len(test_loader.dataset)
    comparison_count = 512 if current_count != 512 else 1024
    comparison_loader = DataLoader(
        Subset(test_data, eval_indices[:comparison_count].tolist()), batch_size=BATCH_SIZE
    )
    hooks = attach_activation_hooks(simulated, activation_cfg, ranges)
    try:
        comparison_accuracy = evaluate(simulated, comparison_loader)
    finally:
        for hook in hooks:
            hook.remove()

    print(f"같은 Q/DQ 모델 | {current_count:,}장 {selected_accuracy:.2f}% | "
          f"{comparison_count:,}장 {comparison_accuracy:.2f}%")
    fig, axis = plt.subplots(figsize=(6, 3), constrained_layout=True)
    bars = axis.bar([f"{current_count:,} images", f"{comparison_count:,} images"],
                    [selected_accuracy, comparison_accuracy], color=["#167d9a", "#777777"])
    axis.bar_label(bars, fmt="%.2f%%", padding=3)
    axis.set(ylim=(0, 108), ylabel="Test accuracy (%)", title="Same Q/DQ model, different test sample counts")
    plt.show()


from IPython.display import Markdown, display

# 셀 9를 완료할 때마다 실제 사용한 옵션과 측정값을 한 행으로 남긴다.
# 같은 Colab 런타임에서 모두 실행으로 다시 시작해도 앞선 행을 유지한다.
if "ptq_history" not in globals():
    ptq_history = []

def activation_bit_ratio(bits):
    return f"{100 * bits / 32:g}%"

ptq_history.append((
    str(len(ptq_history) + 1),
    format_quant_config(weight_cfg),
    format_quant_config(activation_cfg),
    f"{len(calibration_loader.dataset):,} / {len(test_loader.dataset):,}",
    f"{selected_mae:.6f}",
    f"{selected_accuracy:.2f}%",
    f"{selected_accuracy - fp32_accuracy:+.2f}%p",
    activation_bit_ratio(activation_cfg.bits),
))
columns = (
    "회차", "Weight", "Activation", "Calibration / 테스트(장)",
    "첫 ReLU MAE", "테스트 정확도", "FP32 대비 정확도 차이",
    "이론적 Activation 비트량(FP32 대비)",
)
# 이전 버전 행에는 Activation 설정에 저장된 비트 수를 읽어 표시용 열만 채운다.
display_rows = []
for row in ptq_history:
    if len(row) == 7:
        bit_label = row[2].partition(" · ")[0].removesuffix("-bit")
        bit_ratio = activation_bit_ratio(int(bit_label)) if bit_label.isdecimal() else "—"
        row = (*row, bit_ratio)
    display_rows.append(row)
table_rows = [columns, ("---",) * len(columns), *display_rows]
table = "\n".join("| " + " | ".join(row) + " |" for row in table_rows)
display(Markdown("**PTQ 실행 기록**\n\n" + table))